# Session 08 — Vision Transformers

Three hands-on parts mirror the slides:

1. **A simple ViT from scratch** — implement a small Vision Transformer for Fruits-360 and visualise its attention maps.
2. **Pretrained DeiT / Swin** — load a foundation-scale backbone with `timm`, fine-tune just a linear classifier head on Fruits-360.
3. **Self-supervised pretraining** — train a ViT with no labels using a tiny **MAE** (Masked Autoencoder), then linear-probe the resulting features.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/schedldave/cvi4ic-notebooks/blob/main/08-vision-transformers.ipynb)

**Estimated runtime on a Colab T4:** ~25–35 min for everything (Parts 1 + 2 + 3).

> **Runtime → Change runtime type → GPU.** This notebook is much slower on CPU.


## 0. Setup


In [ ]:
!pip install -q timm thop opencv-python-headless tqdm scikit-learn


In [ ]:
import os, math, random, time, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch: {torch.__version__}  ·  Device: {device}")


## 1. Dataset — Fruits-360, grouped to 10 super-classes

Same dataset as session 07 — 10 super-classes (Apple, Banana, …, Tomato).


In [ ]:
!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null || echo "(already cloned)"
DATA_ROOT = Path("fruits-360-100x100")
TRAIN_DIR = DATA_ROOT / "Training"
TEST_DIR  = DATA_ROOT / "Test"


In [ ]:
CLASS_GROUPS = {
    "Apple":    ["Apple Braeburn 1", "Apple Golden 1", "Apple Granny Smith 1",
                 "Apple Pink Lady 1", "Apple Red Delicious 1"],
    "Banana":   ["Banana 1", "Banana 3", "Banana 4", "Banana Lady Finger 1", "Banana Red 1"],
    "Cherry":   ["Cherry 1", "Cherry Rainier 1", "Cherry Sour 1", "Cherry Wax Black 1", "Cherry Wax Yellow 1"],
    "Cucumber": ["Cucumber 1", "Cucumber 3", "Cucumber 5", "Cucumber 7", "Cucumber 9"],
    "Grape":    ["Grape 1", "Grape Blue 1", "Grape Pink 1", "Grape White 1", "Grape White 2"],
    "Peach":    ["Peach 1", "Peach 2", "Peach 3", "Peach 5", "Peach Flat 1"],
    "Pear":     ["Pear Abate 1", "Pear Forelle 1", "Pear Kaiser 1", "Pear Red 1", "Pear Williams 1"],
    "Pepper":   ["Pepper 1", "Pepper Green 1", "Pepper Orange 1", "Pepper Red 1", "Pepper Yellow 1"],
    "Plum":     ["Plum 1", "Plum 2", "Plum 3", "Plum 4", "Plum 5"],
    "Tomato":   ["Tomato 1", "Tomato Cherry Red 1", "Tomato Heart 1", "Tomato Maroon 1", "Tomato Yellow 1"],
}
CLASS_NAMES = sorted(CLASS_GROUPS.keys())
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 96    # multiple of patch_size 16 → 6×6 = 36 patches

NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

class GroupedFruits(Dataset):
    FOLDER_TO_GROUP = {folder: i for i, name in enumerate(CLASS_NAMES)
                       for folder in CLASS_GROUPS[name]}
    def __init__(self, root, transform, samples_per_class=None):
        self.transform = transform
        self.samples = []
        for folder, group_idx in self.FOLDER_TO_GROUP.items():
            cls_dir = root / folder
            if not cls_dir.is_dir(): continue
            files = sorted(cls_dir.glob("*.jpg"))
            if samples_per_class is not None:
                share = max(1, samples_per_class // 5)
                files = files[:share]
            self.samples.extend((f, group_idx) for f in files)
        random.Random(SEED).shuffle(self.samples)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), NORMALIZE,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), NORMALIZE,
])
train_ds = GroupedFruits(TRAIN_DIR, train_tf, samples_per_class=750)
test_ds  = GroupedFruits(TEST_DIR,  eval_tf,  samples_per_class=250)

BATCH = 64
pin = (device.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}")


In [ ]:
def denorm(img_t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (img_t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


## 2. Part A — A simple ViT from scratch

We implement the smallest useful ViT and train it on Fruits-360. Same building blocks as the slide:

```
image (3, 96, 96)
  └── PatchEmbed:  Conv2d(3, dim, k=16, s=16)            → 6×6 = 36 patch tokens
  └── prepend CLS token, add learned positional embedding
  └── N × TransformerBlock (LN → MSA → LN → MLP)
  └── take CLS, LayerNorm, Linear(dim, num_classes)
```


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads=6):
        super().__init__()
        self.h = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, return_attn=False):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                          # (B, h, N, d_h)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, N, D)
        out  = self.proj(out)
        return (out, attn) if return_attn else out


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadSelfAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class SimpleViT(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch_size=16, dim=192,
                 depth=6, num_heads=6, num_classes=NUM_CLASSES):
        super().__init__()
        assert img_size % patch_size == 0
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        n_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(dim, num_heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(dim)
        self.head   = nn.Linear(dim, num_classes)

    def forward(self, x, return_attn=False):
        B = x.size(0)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)        # (B, N, dim)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = x + self.pos_embed
        attn_maps = []
        for blk in self.blocks:
            if return_attn:
                # use the same blk's attn but capture
                a, attn = blk.attn(blk.norm1(x), return_attn=True)
                x = x + a
                x = x + blk.mlp(blk.norm2(x))
                attn_maps.append(attn)
            else:
                x = blk(x)
        x = self.norm(x)
        logits = self.head(x[:, 0])
        return (logits, attn_maps) if return_attn else logits

vit = SimpleViT().to(device)
n_params = sum(p.numel() for p in vit.parameters())
print(f"SimpleViT — {n_params:,} params")


In [ ]:
def train_one(model, loaders, epochs=8, lr=3e-4, wd=1e-4, label="model"):
    train_loader, test_loader = loaders
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss()
    hist = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}
    for ep in range(1, epochs + 1):
        model.train(); rl=rc=rt=0
        for x, y in tqdm(train_loader, desc=f"[{label}] ep {ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = crit(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            rl += loss.item() * x.size(0); rc += (logits.argmax(1) == y).sum().item(); rt += x.size(0)
        sched.step()
        train_loss, train_acc = rl/rt, rc/rt
        model.eval(); tl=tc=tt=0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device); logits = model(x)
                tl += crit(logits, y).item() * x.size(0); tc += (logits.argmax(1) == y).sum().item(); tt += x.size(0)
        test_loss, test_acc = tl/tt, tc/tt
        hist["train_acc"].append(train_acc); hist["test_acc"].append(test_acc)
        hist["train_loss"].append(train_loss); hist["test_loss"].append(test_loss)
        print(f"  ep {ep:2d}/{epochs}  train={train_acc:.3f}  test={test_acc:.3f}  loss={train_loss:.3f}/{test_loss:.3f}")
    return hist

hist_vit = train_one(vit, (train_loader, test_loader), epochs=8, label="SimpleViT")


### Visualising attention — three strategies

Each block produces an attention tensor of shape `(B, heads, N+1, N+1)` — for ViT-B that's 12·197² ≈ 466 k numbers per layer. We need to compress before we can look. Three standard strategies:

1. **CLS-token attention** — take row 0 (the CLS-token's attention to every patch), reshape to the patch grid → one heatmap per head per layer.
2. **Head aggregation** — combine the *h* heads: single head, mean, or max over heads.
3. **Layer aggregation** — plot per-block, *or* use **attention rollout** to compose attention through the whole stack.


In [ ]:
# Pick a sample image and capture attention from every block
img, label = test_ds[random.randint(0, len(test_ds) - 1)]
vit.eval()
with torch.no_grad():
    logits, attn_maps = vit(img.unsqueeze(0).to(device), return_attn=True)
pred = logits.argmax(-1).item()
print(f"true: {CLASS_NAMES[label]}   ·   predicted: {CLASS_NAMES[pred]}")

n_grid = IMG_SIZE // 16          # 6
fig, axes = plt.subplots(1, len(attn_maps) + 1, figsize=(3 * (len(attn_maps) + 1), 3))
axes[0].imshow(denorm(img)); axes[0].set_title("input"); axes[0].axis("off")
for i, attn in enumerate(attn_maps):
    # Average over heads, take CLS row, drop the CLS column → (n_patches,)
    cls_attn = attn[0].mean(0)[0, 1:].reshape(n_grid, n_grid).cpu().numpy()
    axes[i + 1].imshow(cls_attn, cmap="viridis")
    axes[i + 1].set_title(f"block {i+1}"); axes[i + 1].axis("off")
plt.suptitle("CLS-token attention per block — averaged across heads")
plt.tight_layout(); plt.show()


**Strategy 2a — Per head, single layer.** Each head can specialise (some attend to backgrounds, some to centre, some to specific colours). Plot the 6 heads of a chosen layer side by side.


In [ ]:
# Pick a layer and display every head separately
LAYER = -1   # last block — usually the most semantic
attn = attn_maps[LAYER][0]                  # (heads, N+1, N+1)
n_grid = IMG_SIZE // 16
h = attn.shape[0]
fig, axes = plt.subplots(1, h + 1, figsize=(3 * (h + 1), 3))
axes[0].imshow(denorm(img)); axes[0].set_title("input"); axes[0].axis("off")
for i in range(h):
    cls_attn = attn[i, 0, 1:].reshape(n_grid, n_grid).cpu().numpy()
    axes[i + 1].imshow(cls_attn, cmap="viridis")
    axes[i + 1].set_title(f"head {i}"); axes[i + 1].axis("off")
plt.suptitle(f"Block {LAYER} — CLS-token attention per head")
plt.tight_layout(); plt.show()


**Strategy 2b — Three aggregations at the deepest layer.** Compare *single head*, *mean over heads*, *max over heads* on the same block.


In [ ]:
attn = attn_maps[-1][0]                     # last block, drop batch → (heads, N+1, N+1)
cls_per_head = attn[:, 0, 1:].reshape(attn.shape[0], n_grid, n_grid).cpu().numpy()

single_head = cls_per_head[0]
mean_heads  = cls_per_head.mean(0)
max_heads   = cls_per_head.max(0)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
axes[0].imshow(denorm(img)); axes[0].set_title("input"); axes[0].axis("off")
for ax, m, name in zip(axes[1:], [single_head, mean_heads, max_heads],
                        ["single head 0", "mean over heads", "max over heads"]):
    ax.imshow(m, cmap="viridis"); ax.set_title(name); ax.axis("off")
plt.suptitle("Last block — three head-aggregation strategies")
plt.tight_layout(); plt.show()


**Strategy 3 — Attention rollout (Abnar & Zuidema 2020).** Compose attention through *all* blocks. The rollout matrix at layer L is

$$A_{\text{rollout}} = \prod_{l=1}^{L} \tfrac{1}{2}\bigl(A_l + I\bigr)$$

(with `A_l` averaged over heads). The `+ I` accounts for the residual connection. Reading the CLS row of the result tells us what each *output* CLS effectively attends to in the *input* image — usually a much sharper map than any single layer.


In [ ]:
# Attention rollout (Abnar & Zuidema, ACL 2020)
def attention_rollout(attn_maps):
    """attn_maps: list of (B, heads, N+1, N+1) tensors. Returns rollout matrix (N+1, N+1)."""
    rollout = None
    for attn in attn_maps:
        a = attn[0].mean(0).cpu()                # average over heads → (N+1, N+1)
        a = (a + torch.eye(a.size(0))) / 2       # add identity for residual
        a = a / a.sum(dim=-1, keepdim=True)      # re-normalise rows
        rollout = a if rollout is None else a @ rollout
    return rollout

rollout = attention_rollout(attn_maps)            # (N+1, N+1)
rollout_cls = rollout[0, 1:].reshape(n_grid, n_grid).numpy()

# Compare: last-block-only vs rollout
last_block = attn_maps[-1][0].mean(0)[0, 1:].reshape(n_grid, n_grid).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(denorm(img));    axes[0].set_title("input");                     axes[0].axis("off")
axes[1].imshow(last_block,      cmap="viridis"); axes[1].set_title("last block — mean over heads");      axes[1].axis("off")
axes[2].imshow(rollout_cls,     cmap="viridis"); axes[2].set_title("rollout — composed through all blocks"); axes[2].axis("off")
plt.suptitle("Last-block attention vs. attention rollout")
plt.tight_layout(); plt.show()


## 3. Part B — Pretrained DeiT / Swin via timm

From-scratch ViTs need a *lot* of data. The standard trick: **load a model pretrained on ImageNet** via `timm`, freeze the backbone, swap in a fresh classifier head, fine-tune just the head on Fruits-360.

You can swap the model name to anything in `timm.list_models()`. Try `deit_tiny_patch16_224`, `swin_tiny_patch4_window7_224`, `vit_base_patch16_224`, `convnextv2_tiny`, etc.


In [ ]:
import timm

# Try a few options — pick whichever you want to fine-tune.
MODEL_NAME = "deit_tiny_patch16_224"   # ~5M params, fast on Colab T4
# MODEL_NAME = "swin_tiny_patch4_window7_224"

backbone = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
backbone = backbone.to(device)

# Freeze everything except the new classifier head
for p in backbone.parameters():
    p.requires_grad = False
# timm exposes the head differently per model — `get_classifier()` returns it generically
head = backbone.get_classifier()
for p in head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f"{MODEL_NAME}:  total={total:,}   trainable={trainable:,}   frozen={total-trainable:,}")


In [ ]:
# Pretrained models expect 224×224 input — make a quick set of loaders for that resolution
PRETRAIN_SIZE = 224
train_tf_224 = transforms.Compose([
    transforms.Resize((PRETRAIN_SIZE, PRETRAIN_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), NORMALIZE,
])
eval_tf_224 = transforms.Compose([
    transforms.Resize((PRETRAIN_SIZE, PRETRAIN_SIZE)),
    transforms.ToTensor(), NORMALIZE,
])
train_ds_224 = GroupedFruits(TRAIN_DIR, train_tf_224, samples_per_class=300)   # smaller subset → faster
test_ds_224  = GroupedFruits(TEST_DIR,  eval_tf_224,  samples_per_class=100)
train_loader_224 = DataLoader(train_ds_224, batch_size=32, shuffle=True,  num_workers=0, pin_memory=pin)
test_loader_224  = DataLoader(test_ds_224,  batch_size=32, shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds_224)}  test: {len(test_ds_224)}")

hist_pre = train_one(backbone, (train_loader_224, test_loader_224), epochs=3, lr=1e-3, label=MODEL_NAME)


In [ ]:
# How well do the *frozen* features cluster the classes? Quick t-SNE-free sanity check:
# average pooled feature → 2D PCA scatter, coloured by class.
from sklearn.decomposition import PCA

# `forward_features` returns the pre-head representation in timm models
feats, labels = [], []
backbone.eval()
with torch.no_grad():
    for x, y in test_loader_224:
        f = backbone.forward_features(x.to(device))
        # ViT/DeiT: (B, N, dim) — average over patches; CNNs: (B, C, H, W) — global pool
        if f.dim() == 3:
            f = f.mean(1)
        elif f.dim() == 4:
            f = f.mean([-2, -1])
        feats.append(f.cpu()); labels.append(y)
feats  = torch.cat(feats).numpy()
labels = torch.cat(labels).numpy()

emb = PCA(n_components=2).fit_transform(feats)
plt.figure(figsize=(7, 5.5))
for c in range(NUM_CLASSES):
    m = labels == c
    plt.scatter(emb[m, 0], emb[m, 1], s=18, label=CLASS_NAMES[c], alpha=0.7)
plt.legend(loc="best", fontsize=8, ncol=2)
plt.title(f"Frozen {MODEL_NAME} features — PCA → 2D")
plt.tight_layout(); plt.show()


## 4. Part C — Self-supervised pretraining with a tiny MAE

**Masked Autoencoder** (He et al., CVPR 2022): randomly mask 75 % of patches, train the network to **reconstruct the missing pixels**. No labels, no class supervision — yet the resulting features transfer surprisingly well.

This is a *toy* MAE — small ViT encoder + tiny ViT decoder + pixel MSE loss — meant to show the mechanism end-to-end in ~50 lines of code. The full MAE is essentially the same code with a deeper encoder / better decoder / longer schedule.


In [ ]:
class MaskedViT(nn.Module):
    """Encoder-only ViT that operates on a *subset* of patches (the unmasked ones)."""
    def __init__(self, img_size=IMG_SIZE, patch_size=16, dim=192, depth=6, num_heads=6):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.dim        = dim
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        # learned positional embedding (one per patch — no CLS token here)
        self.pos_embed  = nn.Parameter(torch.zeros(1, self.n_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(dim, num_heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(dim)

    def forward(self, x, keep_idx):
        """x: (B, 3, H, W); keep_idx: (B, K) indices of patches to keep."""
        B = x.size(0)
        # Embed every patch + add pos embed, then gather the KEEP indices only
        toks = self.patch_embed(x).flatten(2).transpose(1, 2)   # (B, N, dim)
        toks = toks + self.pos_embed
        idx_exp = keep_idx.unsqueeze(-1).expand(-1, -1, self.dim)
        kept = torch.gather(toks, 1, idx_exp)                   # (B, K, dim)
        for blk in self.blocks:
            kept = blk(kept)
        return self.norm(kept)


class TinyDecoder(nn.Module):
    """Lightweight decoder: project back to a smaller dim, pad with mask tokens, run a few blocks, predict pixels."""
    def __init__(self, n_patches, patch_size, enc_dim=192, dec_dim=96, depth=2, num_heads=4):
        super().__init__()
        self.dec_dim = dec_dim
        self.proj = nn.Linear(enc_dim, dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, n_patches, dec_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(dec_dim, num_heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(dec_dim)
        self.out    = nn.Linear(dec_dim, patch_size * patch_size * 3)

    def forward(self, kept, keep_idx, n_patches):
        B, K, _ = kept.shape
        kept = self.proj(kept)
        # Build a full sequence of mask tokens, scatter the kept ones back in
        full = self.mask_token.expand(B, n_patches, -1).clone()
        idx_exp = keep_idx.unsqueeze(-1).expand(-1, -1, self.dec_dim)
        full = full.scatter(1, idx_exp, kept)
        full = full + self.pos_embed
        for blk in self.blocks:
            full = blk(full)
        return self.out(self.norm(full))   # (B, n_patches, p*p*3)


class MAE(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch_size=16, mask_ratio=0.75):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.mask_ratio = mask_ratio
        self.encoder = MaskedViT(img_size, patch_size)
        self.decoder = TinyDecoder(self.n_patches, patch_size,
                                   enc_dim=self.encoder.dim, dec_dim=96, depth=2)

    def random_mask(self, batch_size):
        n_keep = int(self.n_patches * (1 - self.mask_ratio))
        # Generate a random permutation per sample, take the first n_keep as keep indices
        rand = torch.rand(batch_size, self.n_patches, device=self.encoder.pos_embed.device)
        perm = rand.argsort(dim=1)
        keep_idx = perm[:, :n_keep].sort(dim=1).values         # sorted for stable gather/scatter
        return keep_idx

    def patchify(self, imgs):
        """(B, 3, H, W) → (B, N, p*p*3)"""
        p = self.patch_size
        B, C, H, W = imgs.shape
        x = imgs.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 3, 5, 1).reshape(B, -1, p * p * C)
        return x

    def unpatchify(self, x, n_per_side):
        """(B, N, p*p*3) → (B, 3, H, W)"""
        p = self.patch_size; B = x.size(0)
        x = x.reshape(B, n_per_side, n_per_side, p, p, 3)
        x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, 3, n_per_side * p, n_per_side * p)
        return x

    def forward(self, imgs):
        B = imgs.size(0)
        keep = self.random_mask(B)
        kept_tokens = self.encoder(imgs, keep)
        pred = self.decoder(kept_tokens, keep, self.n_patches)
        target = self.patchify(imgs)
        # Build a mask: 1 where the patch was masked (i.e. NOT in `keep`)
        mask = torch.ones(B, self.n_patches, device=imgs.device)
        mask.scatter_(1, keep, 0.0)
        loss = ((pred - target) ** 2).mean(-1)                 # per-patch MSE
        loss = (loss * mask).sum() / mask.sum()                # only masked patches
        return loss, pred, mask, keep

mae = MAE().to(device)
print(f"MAE — encoder params: {sum(p.numel() for p in mae.encoder.parameters()):,}   "
      f"decoder params: {sum(p.numel() for p in mae.decoder.parameters()):,}")


In [ ]:
# Self-supervised pretraining loop — no labels, just pixel reconstruction.
opt   = torch.optim.AdamW(mae.parameters(), lr=1.5e-3, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)

mae.train()
for ep in range(1, 11):
    rl = rt = 0
    for x, _ in tqdm(train_loader, desc=f"[MAE] ep {ep}/10", leave=False):
        x = x.to(device)
        loss, *_ = mae(x)
        opt.zero_grad(); loss.backward(); opt.step()
        rl += loss.item() * x.size(0); rt += x.size(0)
    sched.step()
    print(f"  ep {ep:2d}/10   recon_loss={rl/rt:.4f}")


### Visualising the reconstructions

Below: input image · masked input (75 % of patches blanked) · MAE reconstruction. The reconstruction is fuzzy — that's fine; the goal is to learn *useful representations*, not to be a perfect inpainter.


In [ ]:
mae.eval()
n_grid = IMG_SIZE // mae.patch_size
with torch.no_grad():
    x_batch = torch.stack([test_ds[i][0] for i in range(4)]).to(device)
    loss, pred, mask, keep = mae(x_batch)
    recon = mae.unpatchify(pred, n_grid)
    masked_input = mae.patchify(x_batch).clone()
    masked_input[mask.bool()] = 0.0          # blank out the masked patches
    masked_input = mae.unpatchify(masked_input, n_grid)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0, i].imshow(denorm(x_batch[i]));        axes[0, i].set_title("input");   axes[0, i].axis("off")
    axes[1, i].imshow(denorm(masked_input[i]));   axes[1, i].set_title("masked"); axes[1, i].axis("off")
    axes[2, i].imshow(denorm(recon[i]));          axes[2, i].set_title("recon");  axes[2, i].axis("off")
plt.suptitle(f"MAE reconstructions (75 % patch mask)   ·   final loss {loss.item():.4f}", y=1.0)
plt.tight_layout(); plt.show()


### Linear probe — do the SSL features actually transfer?

Take the *frozen* MAE encoder, average the patch tokens to get a single vector per image, then train **only a linear classifier** on the 10 classes. This is the standard way to evaluate self-supervised pretraining.


In [ ]:
# Helper: extract pooled features from the (no-mask) encoder for a whole loader
def extract_features(loader):
    feats, lbls = [], []
    mae.eval()
    keep_all = torch.arange(mae.n_patches, device=device).unsqueeze(0)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            B = x.size(0)
            tok = mae.encoder(x, keep_all.expand(B, -1))   # (B, N, dim)
            feats.append(tok.mean(1).cpu())
            lbls.append(y)
    return torch.cat(feats), torch.cat(lbls)

X_tr, y_tr = extract_features(train_loader)
X_te, y_te = extract_features(test_loader)
print(f"train features: {X_tr.shape}   test features: {X_te.shape}")

# Train a logistic regression on the frozen features
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000, n_jobs=-1)
clf.fit(X_tr.numpy(), y_tr.numpy())
acc = clf.score(X_te.numpy(), y_te.numpy())
print(f"Linear probe accuracy on Fruits-360 test set: {acc*100:.2f}%")

# For comparison: same probe on RAW pixels (sanity baseline)
X_raw_tr = torch.stack([img.flatten() for img, _ in [(train_ds[i][0], train_ds[i][1]) for i in range(min(2000, len(train_ds)))]])
y_raw_tr = torch.tensor([train_ds[i][1] for i in range(len(X_raw_tr))])
X_raw_te = torch.stack([img.flatten() for img, _ in [(test_ds[i][0], test_ds[i][1]) for i in range(min(800, len(test_ds)))]])
y_raw_te = torch.tensor([test_ds[i][1] for i in range(len(X_raw_te))])
clf_raw = LogisticRegression(max_iter=2000, n_jobs=-1).fit(X_raw_tr.numpy(), y_raw_tr.numpy())
print(f"Same probe on RAW pixels (baseline):           {clf_raw.score(X_raw_te.numpy(), y_raw_te.numpy())*100:.2f}%")


## 5. Wrap-up — three trainings on the same dataset


In [ ]:
summary = []
summary.append(("SimpleViT (from scratch)", sum(p.numel() for p in vit.parameters()), max(hist_vit['test_acc'])))
summary.append((f"{MODEL_NAME} (linear probe head)",
                sum(p.numel() for p in backbone.parameters() if p.requires_grad),
                max(hist_pre['test_acc'])))
summary.append(("MAE features + LogReg (zero label-supervision pretraining)",
                sum(p.numel() for p in mae.encoder.parameters()),
                acc))

print(f"{'Approach':60s} {'Trainable':>12s} {'Test acc':>10s}")
print('-' * 86)
for name, p, a in summary:
    print(f"{name:60s} {p:>12,} {a*100:>9.2f}%")


## ✏️ Exercises

1. **More heads, more depth.** Increase `depth` and `num_heads` in `SimpleViT`. At what point do you start *over-fitting* on Fruits-360 with only 5 train epochs?

2. **Try a different pretrained model.** Swap `MODEL_NAME` in Part B for `swin_tiny_patch4_window7_224`, `vit_base_patch16_224`, or `convnextv2_tiny.fcmae_ft_in22k_in1k`. Compare top-1 and frozen-feature PCA quality.

3. **Vary the MAE mask ratio.** The MAE paper found 75 % was optimal — try 50 %, 60 %, 75 %, 85 %. Plot the linear-probe accuracy as a function of mask ratio.

4. *(Bonus)* **Fine-tune the MAE encoder, don't just probe.** Replace the linear probe with a small fine-tune (unfreeze the encoder, lr 1e-4, 3 epochs). Compare to from-scratch SimpleViT — does the SSL pretraining help?


In [ ]:
# Your code here
